# 🏗️ CEM4644 · MP3 — Object detection for construction
## Homework (individual): *Construction machinery: excavators, dump trucks, wheel loaders*

**No coding needed.** Each grey box below is one *step*: click the ▶ (play) button at its left, wait until it finishes, look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

The workshop taught you how a detector works on the PPE photos. This homework puts the same tools on a second problem, **construction machinery**, and asks what changes. Every machine is boxed as *excavator*, *dump truck* or *wheel loader*.

**What you will do (about 75 minutes)**
1. Look at the machinery photos.
2. Score the machinery model on unseen photos and look at its mistakes; compare with the workshop's numbers.
3. Send it photos from the other world.
4. Turn boxes into an equipment count.
5. Train your own detector: how much data does it need?
6. **Your own photos** - the main deliverable of this homework.

**Before you start:** menu *Runtime → Change runtime type → T4 GPU → Save*. Detection works on CPU too, but the training step in Part 5 is much faster with a GPU.

**Dataset:** Construction machinery: excavators, dump trucks, wheel loaders. Sources and licences are listed at the bottom.

In [ ]:
#@title ▶ Step 0 · Run me first (about 2 minutes) { display-mode: "form" }
#@markdown Click ▶ and wait for the green ✅ line. This downloads the photos and the course model and installs the detection library.
#@markdown If Colab asks whether to run a notebook that was not authored by Google, choose *Run anyway*.
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp3_object_detection", "aec_det"
FOLDERS = ["mp3_object_detection"]           # only this lab folder is downloaded, not the whole course repository

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("sparse-checkout", "set", *FOLDERS)     # also trims a full copy left by an earlier run
            and _git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
    subprocess.run(["git", "-C", REPO, "sparse-checkout", "set", *FOLDERS], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_det import lab
lab.setup(dataset="excavators")


## Part 1 · The machinery photos

Three classes this time - excavator, dump truck, wheel loader - and machines that are large, but often far away, half hidden behind each other, and easy to confuse with one another. Boxes drawn with a **dashed** line are the labels made by people.

In [ ]:
#@title ▶ Step 1a · Browse the labeled photos { display-mode: "form" }
#@markdown Pick a class to see photos that contain it. Untick *with_boxes* to see the raw photos. Click ▶ again for a new selection.
category = "all" #@param ["all", "excavator", "dump truck", "wheel loader"]
how_many = 6 #@param [3, 6, 9] {type:"raw"}
with_boxes = True #@param {type:"boolean"}
lab.show_gallery(category, how_many, with_boxes)


## Part 2 · The machinery model

The **course model** for this homework was fine-tuned on 2,064 labeled machinery photos. It is scored exactly as in the workshop: on unseen test photos, a detection is **correct** when it has the right class and overlaps the true box by at least half; **recall** is the share of true machines found, **precision** the share of detections that were right, **mAP50** the overall quality score.

In [ ]:
#@title ▶ Step 2a · Score it on all the unseen test photos { display-mode: "form" }
#@markdown Per class: how many true objects were found, how many were missed, how many detections were false alarms.
how_many = "all" #@param ["all", "100"]
threshold = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.1}
lab.evaluate(how_many, threshold)


In [ ]:
#@title ▶ Step 2b · Look at the mistakes { display-mode: "form" }
#@markdown Green = correct, orange = false alarm, red dashed = missed object. Choose the kind of mistake and the class; the worst photos come first.
lab.error_explorer()


> ### 📝 Report question 1
> Which machine does the model find most reliably and which one does it miss most often (give the recall numbers)? Look at the missed and the false ones in Step 2b: what do they have in common? Then compare with the numbers you got in the workshop for the PPE model (your Step 6 there): which of the two datasets is the harder one for a detector, and what makes it harder - the size of the objects, how alike the classes look, how many training photos there were?

## Part 3 · Photos from a different world

A detector only knows the kind of photos it was trained on. Here the photos of **Workers and PPE on construction sites** are given to the machinery model, and to the workshop's PPE model, side by side. Each model can only answer with its own classes.

In [ ]:
#@title ▶ Step 3a · Photos from a different world { display-mode: "form" }
how_many = 3 #@param {type:"slider", min:1, max:6, step:1}
threshold = 0.4 #@param {type:"slider", min:0.1, max:0.9, step:0.1}
lab.domain_shift(how_many, threshold)


> ### 📝 Report question 2
> What does the machinery model make of the PPE photos, and what does the PPE model make of them? Does either model stay silent, or does it label what it sees with the wrong names? What does this tell you about buying a detector that was trained on someone else's photos?

## Part 4 · From boxes to an equipment count

Boxes alone are not a decision. A site manager wants **numbers**: how many machines are on site in each photo, and of which type. The dashboard counts the model's boxes over all test photos and compares them with the labeled truth.

In [ ]:
#@title ▶ Step 4a · Dashboard { display-mode: "form" }
#@markdown Run it several times with different thresholds and compare the numbers.
threshold = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.1}
lab.dashboard(threshold)


> ### 📝 Report question 3
> From the dashboard: how many machines does the AI count in total and how many do the labels contain? On how many photos is the count exactly right? Run it at threshold 0.3 and 0.7 and explain which one you would use for (a) an automatic equipment log that nobody checks and (b) a weekly report that a person reads.

## Part 5 · How much data does a detector need?

In the workshop you trained a detector once. Here you run an **experiment**: the same number of passes, more and more training photos, and one run from a random start. Each run takes one to two minutes on a GPU. Suggested ladder: 60 photos · 10 passes → 120 · 10 → all · 10 → then the best of those with a *random* start. Name every run so the leaderboard stays readable.

In [ ]:
#@title ▶ Step 5a · Train { display-mode: "form" }
#@markdown Choose the settings, name the run, click ▶. The quality score (mAP50) on the validation photos is printed after every pass; the final score on the unseen test photos is printed at the end.
training_photos = "60" #@param ["60", "120", "all"]
passes = 10 #@param {type:"slider", min:3, max:15, step:1}
start = "pretrained" #@param ["pretrained", "random"]
run_name = "60 photos" #@param {type:"string"}
n = 10**9 if training_photos == "all" else int(training_photos)
lab.train_my_model(n, passes, start, run_name)


In [ ]:
#@title ▶ Step 5b · Leaderboard { display-mode: "form" }
#@markdown All your runs, best first. Copy this table into your report.
lab.leaderboard()


In [ ]:
#@title ▶ Step 5c · Your best model vs. the course model on the tricky photos { display-mode: "form" }
group = "all" #@param ["all", "hard_real", "crowded", "synthetic", "other_domain", "out_of_scope"]
threshold = 0.5 #@param {type:"slider", min:0.1, max:0.9, step:0.1}
lab.compare_my_model(group, threshold)


> ### 📝 Report question 4
> Copy your leaderboard. How does the quality score grow with the number of training photos, and where does it start to flatten? How far is your best run from the course model, and what would it take to close the gap? What did the random start do, and what does that tell you about the 120,000 everyday photos the pretrained model had already seen?

## Part 6 · Your own photos

This is the main deliverable. Find 5 photos of construction machinery of your own (photos of construction machinery (from a site you can access, from the street, or photos you find online)) and put them through the machinery model.

In [ ]:
#@title ▶ Step 6a · Your own photos { display-mode: "form" }
#@markdown This cell prints a **link**: open it in a new tab (or on your phone). The app has two tabs. *Photo*: upload photos of construction machinery (from a site you can access, from the street, or photos you find online). *Live camera*: allow the camera and point it at a machine, a toy, or a picture on a screen; boxes update about once a second. Take screenshots for your report.
lab.upload_app()


> ### 📝 Report question 5
> Test 5 photos of your own in Step 6a. Include the screenshots. For each photo: what the model found, what it should have found, and what made the wrong ones hard (distance, angle, a machine the classes do not cover, a photo unlike the training photos). Which two of your photos would you add to the training set, and why those?

## Wrap-up

In [ ]:
#@title ▶ Step 7 · Numbers for your report { display-mode: "form" }
lab.report_summary()


### Data and model sources
- **Construction machinery: excavators, dump trucks, wheel loaders** — Photos of earth-moving equipment on sites and roads, each machine boxed as excavator, dump truck or wheel loader. Source: Roboflow 100 'excavators' benchmark set, CC-BY-4.0.
- **Workers and PPE on construction sites** (used in Step 3a) — Photos of construction sites with every worker boxed as 'person', plus a box for the head (helmet or NO helmet) and the torso (vest or NO vest). Source: Roboflow 100 'construction-safety' benchmark set, CC-BY-4.0.
- Detector: YOLO11n by Ultralytics (AGPL-3.0), pretrained on COCO, fine-tuned for this course. The `ultralytics` library is installed in Step 0.
- Out-of-scope sample images: scikit-image data (public domain / CC0).
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp3_object_detection`).